In [1]:
import arxiv 
import sys 
from typing import List 
import pprint
from pathlib import Path 
import os 
import anthropic
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv()

## Metadata

In [2]:
PAPER_DIR = "papers"
INFO_FILE_NAME = 'papers.json'

## Define llm

In [3]:
llm = anthropic.Anthropic()

## Add function for searching the paper 

In [4]:
def search_papers(topic: str, max_results: int = 10) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information. 

    Args: 
        topic: The topic to search for
        max_results: Maximum number of results to retrieve 

    Returns: 
        List of paper IDs found in the search
    """
    # Define client 
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    new_papers_info = {
        p.get_short_id(): {
            'title': p.title, 
            'authors': [author.name for author in p.authors], 
            'summary': p.summary, 
            'published': str(p.published.date())
        }

        for p in papers
    }
    
    # Save this info along with other papers    
    dir_name = Path(PAPER_DIR)/ topic.lower().replace(" ", "_")
    file_name = Path(dir_name) / INFO_FILE_NAME
    os.makedirs(dir_name, exist_ok=True)

    # Read if there is an existing search on this topic 
    try: 
        with open(file_name, 'r') as json_file: 
            existing_papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError): 
        existing_papers_info = {}

    # Combine existing and new papers
    combined_papers_info = {**existing_papers_info, **new_papers_info}

    # Write into JSON file
    with open(file_name, 'w') as json_file: 
        json.dump(combined_papers_info, json_file, indent=2)

    print(f"Results updated in the file:\n{str(file_name)}")

    return list(new_papers_info.keys())



In [5]:
search_papers("Manny")

Results updated in the file:
papers/manny/papers.json


['1908.02121v1',
 '1908.06877v1',
 '1907.13214v1',
 '2208.06882v2',
 '1510.01942v1',
 '2601.12020v1',
 '1103.5433v6',
 'cmp-lg/9505045v1',
 'cmp-lg/9405024v1',
 'cmp-lg/9407010v1']

## Add function for extracting the paper info for a given paper ID

In [6]:
def extract_info(paper_id: str) -> dict:
    """
    Search all topic folders for information about a specific paper.

    Args:
        paper_id: The arXiv ID of the paper to look for (e.g. '1908.02121v1')

    Returns:
        Dictionary with the paper's title, authors, summary and published date,
        or an empty dictionary if the paper isn't found in any saved search
    """
    for p in Path(PAPER_DIR).iterdir():
        if p.is_dir():
            file_path = p / INFO_FILE_NAME
            try: 
                with open(file_path, 'r') as json_file: 
                    file_contents = json.load(json_file)
                return json.dumps(file_contents[paper_id], indent=2)
            except (FileNotFoundError, json.JSONDecodeError, KeyError): 
                pass
    return {}

pprint.pp(extract_info('1907.13214v1'))

('{\n'
 '  "title": "Alexa as a CALL platform for children: Where do we start?",\n'
 '  "authors": [\n'
 '    "Nikos Tsourakis",\n'
 '    "Manny Rayner",\n'
 '    "Hanieh Habibi",\n'
 '    "Pierre-Emmanuel Gallais",\n'
 '    "Cathy Chua",\n'
 '    "Matt Butterweck"\n'
 '  ],\n'
 '  "summary": "Amazon\'s Alexa is now widely available and shows interesting '
 'potential as a platform for hosting CALL games aimed at children. In this '
 'paper, we describe an initial informal experiment where we created some '
 'simple CALL games and made them available to a few child testers. We report '
 "the children's and parents' reactions. Our overall conclusion is that, "
 'although Alexa has many positive features, there are still fundamental '
 'platform issues in the current version that make it very difficult to build '
 'compelling CALL games for children. The games used will soon be freely '
 'available for download on the Alexa store.",\n'
 '  "published": "2019-07-30"\n'
 '}')


## Process Query

In [7]:
def process_query(query: str):
    messages = [
        {
            'role': 'user', 
            'content': query
        }
    ]

    response = llm.messages.create(
        max_tokens=2024, 
        model='claude-sonnet-4-6',
        messages=messages
    )

    print(response)

    is_query_processed = True
    while is_query_processed: 
        has_tool_usage = False 


        

    

## Chat loop 

In [ ]:
def run_chat_loop():
    print("Type your query below or enter 'quit' to quit")
    while True: 
        try: 
            query = input("Please enter your query: ")
            if query.lower().strip() == 'quit':
                break 
            process_query(query)
            print("Iteration completed")
        except Exception as e: 
            print(f"Error processing query: {str(e)}")

run_chat_loop()

Type your query below or enter 'quit' to quit


Please enter your query:  What time is it?


Message(id='msg_011CgQQQ26yfuFCMjDossAXf', container=None, content=[TextBlock(citations=None, text="I don't have access to real-time information like the current time. You can check the time on your device's clock, or ask a voice assistant like Siri or Google Assistant for the current time. Is there anything else I can help you with?", type='text')], diagnostics=None, model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=12, output_tokens=56, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'))
